# RAG With Dockerized Postgres And pgvector

Retrieval-augmented generation often fails locally because infrastructure setup takes longer than the retrieval logic itself. This notebook shows a package-first path where `py-dockerdb` provisions PostgreSQL with `pgvector`, then `LlamaIndex` uses that database as the vector backend.

This approach matters when you want a reproducible RAG baseline for experimentation, onboarding, or CI without manually installing and tuning a local Postgres stack. By the end, you will have started a pgvector-backed database, indexed documents through `PGVectorStore`, validated retrieval results, and cleaned up the containerized environment.


## Table Of Contents

- [Table Of Contents](#table-of-contents)
- [Prerequisites](#prerequisites)
- [1. Start PostgreSQL With pgvector](#1-start-postgresql-with-pgvector)
- [2. Verify Database Readiness](#2-verify-database-readiness)
- [3. Create The LlamaIndex PGVector Store](#3-create-the-llamaindex-pgvector-store)
- [4. Index Documents For Retrieval](#4-index-documents-for-retrieval)
- [5. Run And Validate Retrieval](#5-run-and-validate-retrieval)
- [6. Conclusion](#6-conclusion)


## Prerequisites

No environment variables are required for this notebook because it uses demo credentials inside the notebook configuration.

Additional prerequisites:
- Docker Desktop must be running before `create_db()` is called.
- Install project dependencies with the `rag` extras (for example, `pip install -e ".[rag]"`).
- Ensure `usage/configs/postgres/Dockerfile.pgvector` and `usage/configs/postgres/initdb_pgvector.sh` exist.


## 1. Start PostgreSQL With pgvector

We begin by loading the exact modules used throughout the workflow so setup, indexing, and retrieval all run against the same runtime context.


In [ ]:
import uuid
import tempfile
from pathlib import Path

import psycopg2

from docker_db.dbs.postgres_db import PostgresConfig, PostgresDB
from llama_index.core import StorageContext, VectorStoreIndex
from llama_index.core.embeddings import MockEmbedding
from llama_index.core.schema import Document
from llama_index.vector_stores.postgres import PGVectorStore

Next, we create a `PostgresConfig` that points to the pgvector Docker image and init script, then start the database with `PostgresDB.create_db()`.

- **workdir**:
  Docker build context used to build the database image.
- **dockerfile_path**:
  Path to the pgvector-enabled Dockerfile used for this container image.
- **init_script**:
  Initialization script mounted into `/docker-entrypoint-initdb.d` to enable `vector` and bootstrap schema.
- **volume_path**:
  Host path used to persist PostgreSQL data files.

This code starts the containerized database and prints the connection string so later RAG steps can rely on a known-good backend.


In [ ]:
temp_dir = Path(tempfile.mkdtemp())

container_name = f"demo-pgvector-{uuid.uuid4().hex[:8]}"
image_name = f"demo-pgvector-image-{uuid.uuid4().hex[:8]}:latest"

usage_root = Path(".").resolve()
if not (usage_root / "configs" / "postgres" / "Dockerfile.pgvector").exists():
    usage_root = usage_root / "usage"

docker_cfg_dir = usage_root / "configs" / "postgres"

config = PostgresConfig(
    user="demouser",
    password="demopass",
    database="vector_db",
    project_name="demo",
    container_name=container_name,
    image_name=image_name,
    port=5432,
    workdir=docker_cfg_dir,
    volume_path=Path(temp_dir, "pgdata_rag").resolve(),
    dockerfile_path=docker_cfg_dir / "Dockerfile.pgvector",
    init_script=docker_cfg_dir / "initdb_pgvector.sh",
    env_vars={"VECTOR_DB_NAME": "vector_db"},
    retries=20,
    delay=3,
)

db_manager = PostgresDB(config)
db_manager.create_db()
print(db_manager.connection_string())

## 2. Verify Database Readiness

Before creating a vector store, we confirm that the `vector` extension is actually installed in the target database.

This validation prevents subtle failures later, because `PGVectorStore` expects `pgvector` operators and types to be available when tables are created and queried.


In [ ]:
conn = psycopg2.connect(
    host=config.host,
    port=config.port,
    user=config.user,
    password=config.password,
    database=config.database,
)
conn.autocommit = True

with conn.cursor() as cur:
    cur.execute("SELECT extname FROM pg_extension WHERE extname = 'vector';")
    print("pgvector extension:", cur.fetchone())

conn.close()

## 3. Create The LlamaIndex PGVector Store

`PGVectorStore` connects LlamaIndex to PostgreSQL so embeddings and metadata are stored in relational tables with vector support.

- **table_name**:
  Table used by the vector store to persist nodes and embeddings.
- **embed_dim**:
  Embedding vector dimensionality that must match the embedding model output.
- **hybrid_search**:
  Enables mixed lexical and vector retrieval behavior in supported modes.

Reference documentation:
- `pgvector` project: https://github.com/pgvector/pgvector
- LlamaIndex Postgres vector store API: https://docs.llamaindex.ai/en/stable/api_reference/storage/vector_store/postgres/
- LlamaIndex Postgres vector store example: https://docs.llamaindex.ai/en/stable/examples/vector_stores/postgres/

Example implementation write-up:
- Medium example (LlamaIndex + pgvector RAG): https://medium.com/@shaikhrayyan123/how-to-build-an-llm-rag-pipeline-with-llama-2-pgvector-and-llamaindex-4494b54eb17d

The next cell initializes the vector store with concrete connection and table settings, which is required before indexing documents.


In [ ]:
db_name = config.database
user = config.user

vector_store = PGVectorStore.from_params(
    database=db_name,
    host=config.host,
    password=config.password,
    port=config.port,
    user=user,
    table_name="physics_docs",
    embed_dim=384,
    hybrid_search=True,
)

vector_store

## 4. Index Documents For Retrieval

With storage configured, we convert raw text into indexable nodes and persist their embeddings in Postgres.

- **Document**:
  Source text unit passed into indexing.
- **MockEmbedding**:
  Lightweight embedding model used here for deterministic local demos.
- **StorageContext**:
  Binds indexing operations to the configured vector store backend.

This code creates a minimal corpus and builds a `VectorStoreIndex`, which prepares data for retrieval queries.


In [ ]:
documents = [
    Document(text="General relativity was developed by Albert Einstein.", metadata={"topic": "physics"}),
    Document(text="The Standard Model describes three fundamental forces.", metadata={"topic": "physics"}),
    Document(text="Docker makes local infrastructure reproducible for development.", metadata={"topic": "devops"}),
]

embed_model = MockEmbedding(embed_dim=384)
storage_context = StorageContext.from_defaults(vector_store=vector_store)

index = VectorStoreIndex.from_documents(
    documents,
    storage_context=storage_context,
    embed_model=embed_model,
    show_progress=False,
)

print(f"Indexed {len(documents)} documents.")

## 5. Run And Validate Retrieval

Now we issue a semantic question and inspect retrieved nodes to verify that the indexed content is queryable.

This retrieval pass is the practical proof that the local RAG data path is working end to end.


In [ ]:
query = "Who developed general relativity?"
retriever = index.as_retriever(similarity_top_k=2)
results = retriever.retrieve(query)

print(f"Retrieved {len(results)} nodes")
for i, node in enumerate(results, start=1):
    print(f"\nResult {i} (score={node.score})")
    print(node.text)

assert len(results) > 0

**Teardown**

The final operational step is to remove the demo container so local resources are released and subsequent runs start from a clean state.


In [ ]:
db_manager.delete_db(running_ok=True)
print(f"Stopped and removed {container_name}")

## 6. Conclusion

You have completed the workflow introduced in the beginning:
1. Started PostgreSQL with `pgvector` through `py-dockerdb`.
2. Created a LlamaIndex `PGVectorStore` on that database.
3. Indexed sample documents and validated retrieval results.
4. Cleaned up the containerized runtime.

Key caveats and next steps:
1. Replace `MockEmbedding` with your production embedding model and keep `embed_dim` aligned.
2. For larger datasets, tune indexing and search options described in the LlamaIndex Postgres example docs.
3. If you persist data volumes between runs, document a reset strategy so experiments remain reproducible.
